# Module 080: Milestone Project: Weather Dashboard CLI

Phase 8: Data, Web & APIs | Duration: 3 hours | Milestone: Yes

## Overview

Build a CLI weather dashboard using the OpenWeatherMap API. This project integrates:
- HTTP requests with `requests`
- JSON parsing and file I/O
- Error handling
- Type hints
- Command-line argument parsing with `argparse`
- Formatted console output

## Step 1: Setting Up the API Request

In [ ]:
import requests
from typing import Dict, Any, Optional

API_KEY: str = "YOUR_API_KEY_HERE"

def build_url(city: str) -> str:
    base: str = "https://api.openweathermap.org/data/2.5/weather"
    return f"{base}?q={city}&appid={API_KEY}&units=metric"

def fetch_weather(city: str) -> Optional[Dict[str, Any]]:
    try:
        resp = requests.get(build_url(city), timeout=10)
        resp.raise_for_status()
        return resp.json()
    except requests.exceptions.RequestException:
        return None

## Step 2: Parsing and Displaying Weather Data

In [ ]:
def parse_weather(data: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "city": data["name"],
        "temperature": data["main"]["temp"],
        "humidity": data["main"]["humidity"],
        "conditions": data["weather"][0]["description"],
        "wind_speed": data["wind"]["speed"],
    }

def display_weather(weather: Dict[str, Any]) -> None:
    print("\n" + "=" * 40)
    print(f"  Weather in {weather['city']}")
    print("=" * 40)
    print(f"  Temperature: {weather['temperature']:.1f}°C")
    print(f"  Humidity:    {weather['humidity']}%")
    print(f"  Conditions:  {weather['conditions'].capitalize()}")
    print(f"  Wind Speed:  {weather['wind_speed']} m/s")
    print("=" * 40 + "\n")

## Step 3: Storing Recent Searches

In [ ]:
import json
from typing import List

def save_search(city: str, filepath: str = "recent.json") -> None:
    try:
        with open(filepath) as f:
            searches: List[str] = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        searches = []
    if city in searches:
        searches.remove(city)
    searches.insert(0, city)
    searches = searches[:5]
    with open(filepath, "w") as f:
        json.dump(searches, f)

def load_searches(filepath: str = "recent.json") -> List[str]:
    try:
        with open(filepath) as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return []

## Step 4: Fetching the 5-Day Forecast

In [ ]:
def build_forecast_url(city: str) -> str:
    base: str = "https://api.openweathermap.org/data/2.5/forecast"
    return f"{base}?q={city}&appid={API_KEY}&units=metric"

def fetch_forecast(city: str) -> Optional[List[Dict[str, Any]]]:
    try:
        resp = requests.get(build_forecast_url(city), timeout=10)
        resp.raise_for_status()
        data = resp.json()
        return [{
            "datetime": entry["dt_txt"],
            "temperature": entry["main"]["temp"],
            "conditions": entry["weather"][0]["description"],
        } for entry in data["list"]]
    except requests.exceptions.RequestException:
        return None

## Step 5: Building the CLI

In [ ]:
import argparse

def main() -> None:
    parser = argparse.ArgumentParser(description="Weather Dashboard CLI")
    parser.add_argument("city", nargs="?", help="City to query")
    parser.add_argument("--forecast", action="store_true", help="Show forecast")
    parser.add_argument("--recent", action="store_true", help="Show recent")
    args = parser.parse_args()

    if args.recent:
        for c in load_searches():
            print(f"  - {c}")
        return

    if not args.city:
        parser.print_help()
        return

    data = fetch_weather(args.city)
    if data:
        display_weather(parse_weather(data))
        save_search(args.city)

    if args.forecast:
        f = fetch_forecast(args.city)
        if f:
            for entry in f[:8]:
                print(f"  {entry['datetime']} | {entry['temperature']:.1f}°C | {entry['conditions']}")

if __name__ == "__main__":
    main()